<a href="https://colab.research.google.com/github/trungbeo789852/XLAS/blob/main/XLAS_10_11_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python ultralytics pandas matplotlib
!pip install -q ultralytics supervision
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
THỐNG KÊ THỜI GIAN & QUÃNG ĐƯỜNG DI CHUYỂN CỦA CẦU THỦ TỪ VIDEO BÓNG ĐÁ
(Tích hợp: Tự động gom cụm nhận diện 2 màu áo chủ đạo + Quy tắc Vùng Khung Thành + ReID Độc quyền 22 ID + Nhãn trong suốt)
"""
import argparse
import os
import shutil
import subprocess
import sys
import time
from collections import deque
from dataclasses import dataclass, field

import cv2
import numpy as np
import pandas as pd

# ============================ THAM SỐ CHÍNH ============================
PLAYER_HEIGHT_M = 1.75
STEP_FRAMES = 6
V_MIN_MS = 0.5
V_MAX_MS = 11.5
SPRINT_KMH = 25.0
MAX_GAP = 15
MIN_TRACK_SEC = 1.0
CUT_CORR_THR = 0.45
GRASS_HSV_LO = (30, 40, 40)
GRASS_HSV_HI = (90, 255, 255)


# ============================ PHÁT HIỆN NGƯỜI ============================
class PersonDetector:
    def __init__(self, model_path="yolov8m.pt", imgsz=1280, conf=0.15,
                 iou=0.5, device=None, tiles=False):
        from ultralytics import YOLO
        self.model = YOLO(model_path)
        self.imgsz, self.conf, self.iou = imgsz, conf, iou
        self.device, self.tiles = device, tiles

    def _infer(self, img, imgsz):
        r = self.model.predict(img, imgsz=imgsz, conf=self.conf, iou=self.iou,
                               classes=[0], device=self.device, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return np.zeros((0, 4), np.float32), np.zeros(0, np.float32)
        return r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()

    def detect(self, frame):
        H, W = frame.shape[:2]
        boxes, scores = self._infer(frame, self.imgsz)
        if not self.tiles:
            return boxes, scores

        tw, th = int(W * 0.6), int(H * 0.6)
        tile_imgsz = int(np.ceil(max(tw, th) * 2 / 32) * 32)
        all_b, all_s = [boxes], [scores]
        for x0 in (0, W - tw):
            for y0 in (0, H - th):
                tb, ts = self._infer(frame[y0:y0 + th, x0:x0 + tw], tile_imgsz)
                if len(tb) == 0:
                    continue
                keep = np.ones(len(tb), bool)
                if x0 > 0:
                    keep &= tb[:, 0] > 3
                if x0 == 0 and tw < W:
                    keep &= tb[:, 2] < tw - 3
                if y0 > 0:
                    keep &= tb[:, 1] > 3
                if y0 == 0 and th < H:
                    keep &= tb[:, 3] < th - 3
                tb, ts = tb[keep], ts[keep]
                tb = tb + np.array([x0, y0, x0, y0], np.float32)
                all_b.append(tb)
                all_s.append(ts)
        b = np.vstack(all_b).astype(np.float32)
        s = np.concatenate(all_s).astype(np.float32)
        if len(b) == 0:
            return b, s
        xywh = [[float(x1), float(y1), float(x2 - x1), float(y2 - y1)] for x1, y1, x2, y2 in b]
        idx = cv2.dnn.NMSBoxes(xywh, s.tolist(), self.conf, 0.5)
        idx = np.array(idx).flatten().astype(int)
        return b[idx], s[idx]


def filter_detections(boxes, scores, frame):
    if len(boxes) == 0:
        return boxes, scores
    H, W = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    grass = cv2.inRange(hsv, GRASS_HSV_LO, GRASS_HSV_HI)
    keep = []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        w, h = x2 - x1, y2 - y1
        if h < 20 or w < 8 or h > 0.8 * H:
            continue
        if h / max(w, 1) < 1.1 or h / max(w, 1) > 4.0:
            continue
        ys, ye = int(max(y2 - 0.05 * h, 0)), int(min(y2 + 0.15 * h + 2, H))
        xs, xe = int(max(x1, 0)), int(min(x2, W))
        if ye <= ys or xe <= xs:
            continue
        if (grass[ys:ye, xs:xe] > 0).mean() >= 0.60:
            keep.append(i)
    keep = np.array(keep, int)
    return boxes[keep], scores[keep]


# ============================ TỰ ĐỘNG PHÂN TÍCH & PHÂN LOẠI MÀU ÁO ĐẤU ĐỘNG ============================
class DynamicTeamColorMatcher:
    def __init__(self):
        self.team_centers = None

    def learn_dominant_colors(self, first_frames_data):
        """
        Phân tích tập hợp tất cả các vùng áo của các đối tượng được phát hiện
        để tìm ra 2 tâm màu sắc nhiều nhất thông qua thuật toán KMeans.
        """
        hues = []
        for shirt_hsv in first_frames_data:
            if shirt_hsv.size == 0:
                continue
            grass_mask = cv2.inRange(shirt_hsv, GRASS_HSV_LO, GRASS_HSV_HI)
            non_grass = shirt_hsv[grass_mask == 0]
            if len(non_grass) > 0:
                hues.extend(non_grass[:, :2])

        if len(hues) < 50:
            return False

        hues = np.array(hues, dtype=np.float32)
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 15, 1.0)
        flags = cv2.KMEANS_RANDOM_CENTERS
        compactness, labels, centers = cv2.kmeans(hues, 2, None, criteria, 10, flags)

        self.team_centers = centers
        print(f"\n[Bộ Nhận Diện Màu Động] Đã học tự động thành công dải màu của 2 đội bóng.", flush=True)
        return True

    def is_player_color(self, shirt_hsv, threshold=45.0):
        if self.team_centers is None:
            return True

        grass_mask = cv2.inRange(shirt_hsv, GRASS_HSV_LO, GRASS_HSV_HI)
        non_grass = shirt_hsv[grass_mask == 0]
        if len(non_grass) < 5:
            return False

        median_color = np.median(non_grass[:, :2], axis=0)
        dist1 = np.linalg.norm(median_color - self.team_centers[0])
        dist2 = np.linalg.norm(median_color - self.team_centers[1])

        return min(dist1, dist2) < threshold


def filter_players_by_dynamic_color(frame, boxes, color_matcher):
    if len(boxes) == 0:
        return np.ones(0, dtype=bool)

    keep_mask = []
    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    H, W = frame.shape[:2]

    goal_left_x_max = W * 0.18
    goal_right_x_min = W * 0.82

    for (x1, y1, x2, y2) in boxes:
        w, h = x2 - x1, y2 - y1
        cx1, cx2 = int(x1 + w * 0.2), int(x2 - w * 0.2)
        cy1, cy2 = int(y1 + h * 0.1), int(y1 + h * 0.5)

        cy1, cy2 = max(0, cy1), max(0, cy2)
        cx1, cx2 = max(0, cx1), max(0, cx2)

        shirt_hsv = hsv_frame[cy1:cy2, cx1:cx2]

        if color_matcher.is_player_color(shirt_hsv):
            keep_mask.append(True)
        else:
            center_x = (x1 + x2) / 2.0
            if center_x <= goal_left_x_max or center_x >= goal_right_x_min:
                keep_mask.append(True)
            else:
                keep_mask.append(False)

    return np.array(keep_mask, dtype=bool)


# ============================ QUẢN LÝ ĐỊNH DANH (BỘ NHỚ KHÔNG GIAN) ============================
class IdentityManager:
    def __init__(self, max_players=22, match_thresh=0.55):
        self.max_players = max_players
        self.match_thresh = match_thresh
        self.bt_to_global = {}
        self.global_features = {}
        self.global_positions = {}
        self.next_id = 1

    def extract_feature(self, frame, box):
        x1, y1, x2, y2 = map(int, box)
        roi = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        if roi.size == 0:
            return None
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
        cv2.normalize(hist, hist)
        return hist

    def get_center(self, box):
        return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

    def process_frame(self, frame, bt_ids, boxes):
        active_global_ids = set()
        results = []
        unmatched_detections = []

        for bt_id, box in zip(bt_ids, boxes):
            feat = self.extract_feature(frame, box)
            center = self.get_center(box)
            if feat is None:
                continue

            if bt_id in self.bt_to_global:
                gid = self.bt_to_global[bt_id]
                if gid not in active_global_ids:
                    old_feat = self.global_features[gid]
                    self.global_features[gid] = cv2.addWeighted(old_feat, 0.9, feat, 0.1, 0)
                    self.global_positions[gid] = center
                    active_global_ids.add(gid)
                    results.append((bt_id, gid))
                    continue

            unmatched_detections.append((bt_id, feat, center, box))

        for bt_id, feat, center, box in unmatched_detections:
            best_match_id = None
            best_score = -1

            for gid, gfeat in self.global_features.items():
                if gid in active_global_ids:
                    continue

                app_score = cv2.compareHist(feat, gfeat, cv2.HISTCMP_CORREL)

                last_pos = self.global_positions.get(gid, center)
                dist = np.hypot(center[0] - last_pos[0], center[1] - last_pos[1])
                spatial_penalty = min(0.6, dist / 400.0)

                total_score = app_score - spatial_penalty

                if total_score > best_score:
                    best_score = total_score
                    best_match_id = gid

            if best_match_id is not None and best_score >= self.match_thresh:
                self.bt_to_global[bt_id] = best_match_id
                active_global_ids.add(best_match_id)
                self.global_positions[best_match_id] = center
                results.append((bt_id, best_match_id))
            else:
                if len(self.global_features) < self.max_players:
                    gid = self.next_id
                    self.next_id += 1
                    self.bt_to_global[bt_id] = gid
                    self.global_features[gid] = feat
                    self.global_positions[gid] = center
                    active_global_ids.add(gid)
                    results.append((bt_id, gid))
                else:
                    results.append((bt_id, None))

        return {r[0]: r[1] for r in results}


# ============================ CẮT CẢNH ====================
class CutDetector:
    def __init__(self, thr=CUT_CORR_THR):
        self.thr, self.prev = thr, None

    def update(self, frame):
        small = cv2.resize(frame, (160, 90))
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        h = cv2.calcHist([hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
        cv2.normalize(h, h)
        cut = False
        if self.prev is not None:
            cut = cv2.compareHist(self.prev, h, cv2.HISTCMP_CORREL) < self.thr
        self.prev = h
        return cut


# ============================ BÙ CHUYỂN ĐỘNG CAMERA ============================
class CameraMotion:
    def __init__(self, W, H):
        self.W, self.H = W, H
        self.static_mask = np.full((H, W), 255, np.uint8)
        self.static_mask[:int(0.25 * H), int(0.86 * W):] = 0
        self.static_mask[:int(0.22 * H), :int(0.32 * W)] = 0
        self.prev_gray = None
        self.prev_boxes = np.zeros((0, 4), np.float32)
        self.C = np.eye(3)
        self.last_M = np.eye(3)
        self.fail_streak = 0

    def reset(self):
        self.C = np.eye(3)
        self.last_M = np.eye(3)
        self.prev_gray = None
        self.prev_boxes = np.zeros((0, 4), np.float32)
        self.fail_streak = 0

    def _mask(self, boxes):
        m = self.static_mask.copy()
        for x1, y1, x2, y2 in boxes:
            m[max(int(y1) - 8, 0):int(y2) + 8, max(int(x1) - 8, 0):int(x2) + 8] = 0
        return m

    def update(self, frame, boxes):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        ok = True
        M = None
        if self.prev_gray is not None:
            allb = np.vstack([self.prev_boxes, boxes]) if len(boxes) else self.prev_boxes
            p0 = cv2.goodFeaturesToTrack(self.prev_gray, 500, 0.005, 7,
                                         mask=self._mask(allb), blockSize=7)
            ok = False
            if p0 is not None and len(p0) >= 12:
                p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev_gray, gray, p0, None,
                                                     winSize=(21, 21), maxLevel=3)
                g = st.ravel() == 1
                if g.sum() >= 12:
                    A, inl = cv2.estimateAffinePartial2D(p0[g], p1[g], method=cv2.RANSAC,
                                                         ransacReprojThreshold=2.0)
                    if A is not None and inl is not None and int(inl.sum()) >= 10:
                        M = np.eye(3)
                        M[:2] = A
                        ok = True
            if ok:
                self.fail_streak = 0
                self.last_M = M
            else:
                self.fail_streak += 1
                M = self.last_M
            self.C = self.C @ np.linalg.inv(M)
        self.prev_gray = gray
        self.prev_boxes = boxes.copy() if len(boxes) else np.zeros((0, 4), np.float32)
        return ok

    @staticmethod
    def _apply(T, pts):
        p = np.hstack([pts, np.ones((len(pts), 1))]) @ T.T
        return p[:, :2] / p[:, 2:3]

    def boxes_to_ref(self, boxes):
        return self._boxes(self.C, boxes)

    def boxes_from_ref(self, boxes):
        return self._boxes(np.linalg.inv(self.C), boxes)

    def _boxes(self, T, boxes):
        if len(boxes) == 0:
            return boxes.astype(np.float32)
        out = np.zeros_like(boxes, dtype=np.float32)
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            c = self._apply(T, np.array([[x1, y1], [x2, y1], [x1, y2], [x2, y2]], float))
            out[i] = [c[:, 0].min(), c[:, 1].min(), c[:, 0].max(), c[:, 1].max()]
        return out


# ============================ THỐNG KÊ TỪNG TRACK ============================
@dataclass
class PlayerTrack:
    shot: int
    tid: int
    first_frame: int
    last_frame: int = 0
    frames_seen: int = 0
    dist_m: float = 0.0
    time_moving: float = 0.0
    time_still: float = 0.0
    sprint_time: float = 0.0
    max_speed_ms: float = 0.0
    speed_ms: float = 0.0
    anchor: tuple = None
    heights: deque = field(default_factory=lambda: deque(maxlen=45))
    upright: deque = field(default_factory=lambda: deque(maxlen=45))

    def px_per_m(self):
        hs = self.upright if len(self.upright) >= 5 else self.heights
        if len(hs) == 0:
            return None
        return max(float(np.median(hs)) / PLAYER_HEIGHT_M, 5.0)

    def update(self, idx, box_ref, box_frame, frame_wh, fps):
        W, H = frame_wh
        self.frames_seen += 1
        self.last_frame = idx
        x1, y1, x2, y2 = box_ref
        w, h = x2 - x1, y2 - y1
        fx1, fy1, fx2, fy2 = box_frame
        foot_visible = fy2 < H - 2
        height_valid = foot_visible and fy1 > 2
        if height_valid:
            self.heights.append(h)
            if h / max(w, 1e-6) >= 1.6:
                self.upright.append(h)
        if not foot_visible:
            return
        foot = ((x1 + x2) / 2.0, y2)
        if self.anchor is None:
            self.anchor = (idx, *foot)
            return
        a_f, ax, ay = self.anchor
        gap = idx - a_f
        if gap > MAX_GAP + STEP_FRAMES:
            self.anchor = (idx, *foot)
            return
        if gap < STEP_FRAMES:
            return
        scale = self.px_per_m()
        self.anchor = (idx, *foot)
        if scale is None:
            return
        d_m = float(np.hypot(foot[0] - ax, foot[1] - ay)) / scale
        dt = gap / fps
        v = d_m / dt
        if v > V_MAX_MS:
            return
        if v >= V_MIN_MS:
            self.dist_m += d_m
            self.time_moving += dt
        else:
            self.time_still += dt
        self.speed_ms = v if self.speed_ms == 0 else 0.6 * self.speed_ms + 0.4 * v
        self.max_speed_ms = max(self.max_speed_ms, v)
        if v * 3.6 >= SPRINT_KMH:
            self.sprint_time += dt


# ============================ TRACKER (TÍCH HỢP BYTETRACK NỘI BỘ) ============================
class SimpleDetections:
    def __init__(self, xyxy, confidence=None, class_id=None, tracker_id=None):
        self.xyxy = np.asarray(xyxy, dtype=np.float32).reshape(-1, 4)
        n = len(self.xyxy)
        self.confidence = np.asarray(confidence, dtype=np.float32) if confidence is not None else np.ones(n, dtype=np.float32)
        self.class_id = np.asarray(class_id, dtype=int) if class_id is not None else np.zeros(n, dtype=int)
        self.tracker_id = np.asarray(tracker_id, dtype=int) if tracker_id is not None else None

    def __len__(self):
        return len(self.xyxy)


def _iou_matrix(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 1e-6, inter / union, 0.0)


def _linear_match(iou_mat, iou_thresh):
    if iou_mat.size == 0:
        return [], list(range(iou_mat.shape[0])), list(range(iou_mat.shape[1]))
    matches = []
    used_r, used_c = set(), set()
    # Khớp tham lam theo thứ tự IoU giảm dần
    flat_indices = np.argsort(-iou_mat.ravel())
    rows, cols = iou_mat.shape
    for idx in flat_indices:
        r, c = divmod(int(idx), cols)
        if iou_mat[r, c] < iou_thresh:
            break
        if r not in used_r and c not in used_c:
            used_r.add(r)
            used_c.add(c)
            matches.append((r, c))
    unmatched_r = [r for r in range(rows) if r not in used_r]
    unmatched_c = [c for c in range(cols) if c not in used_c]
    return matches, unmatched_r, unmatched_c


class BuiltinByteTrack:
    """Bộ theo dõi ByteTrack nội bộ tương thích hoàn toàn với giao diện của supervision."""
    def __init__(self, track_thresh=0.45, track_buffer=900, match_thresh=0.8):
        self.track_thresh = track_thresh
        self.track_buffer = track_buffer
        self.iou_thresh_high = max(0.15, 1.0 - match_thresh)
        self.iou_thresh_low = 0.35
        self.tracks = []  # Mỗi phần tử: dict(tid, box, vel, lost, conf, cls)
        self.next_id = 1

    def _predict_box(self, tr):
        step = min(tr["lost"] + 1, 5)
        return tr["box"] + tr["vel"] * step

    def update_with_detections(self, detections):
        boxes = detections.xyxy
        confs = detections.confidence
        clss = detections.class_id

        if len(boxes) == 0:
            for tr in self.tracks:
                tr["lost"] += 1
            self.tracks = [tr for tr in self.tracks if tr["lost"] <= self.track_buffer]
            return SimpleDetections(
                np.zeros((0, 4), np.float32),
                np.zeros(0, np.float32),
                np.zeros(0, int),
                np.zeros(0, int)
            )

        high_idx = np.where(confs >= self.track_thresh)[0]
        low_idx = np.where((confs >= 0.1) & (confs < self.track_thresh))[0]

        high_boxes = boxes[high_idx]
        low_boxes = boxes[low_idx]

        pred_boxes = np.array([self._predict_box(tr) for tr in self.tracks], dtype=np.float32) if self.tracks else np.zeros((0, 4), np.float32)

        # Bước 1: Khớp các track hiện có với detection có confidence cao
        iou_high = _iou_matrix(pred_boxes, high_boxes)
        matches_1, unm_tracks_1, unm_high = _linear_match(iou_high, self.iou_thresh_high)

        active_outputs = []

        for t_i, d_i in matches_1:
            tr = self.tracks[t_i]
            new_box = high_boxes[d_i]
            dt = max(1, tr["lost"] + 1)
            tr["vel"] = 0.7 * tr["vel"] + 0.3 * ((new_box - tr["box"]) / dt)
            tr["box"] = new_box
            tr["lost"] = 0
            tr["conf"] = confs[high_idx[d_i]]
            tr["cls"] = clss[high_idx[d_i]]
            active_outputs.append(tr)

        # Bước 2: Khớp các track vừa hoạt động nhưng chưa khớp với detection có confidence thấp
        rem_tracks_idx = [t_i for t_i in unm_tracks_1 if self.tracks[t_i]["lost"] == 0]
        lost_tracks_idx = [t_i for t_i in unm_tracks_1 if self.tracks[t_i]["lost"] > 0]

        rem_pred_boxes = np.array([self._predict_box(self.tracks[t_i]) for t_i in rem_tracks_idx], dtype=np.float32) if rem_tracks_idx else np.zeros((0, 4), np.float32)
        iou_low = _iou_matrix(rem_pred_boxes, low_boxes)
        matches_2, unm_rem_tracks, _ = _linear_match(iou_low, self.iou_thresh_low)

        for local_t_i, d_i in matches_2:
            t_i = rem_tracks_idx[local_t_i]
            tr = self.tracks[t_i]
            new_box = low_boxes[d_i]
            tr["vel"] = 0.7 * tr["vel"] + 0.3 * (new_box - tr["box"])
            tr["box"] = new_box
            tr["lost"] = 0
            tr["conf"] = confs[low_idx[d_i]]
            tr["cls"] = clss[low_idx[d_i]]
            active_outputs.append(tr)

        # Đánh dấu các track không khớp là mất dấu tạm thời
        for local_t_i in unm_rem_tracks:
            self.tracks[rem_tracks_idx[local_t_i]]["lost"] += 1
        for t_i in lost_tracks_idx:
            self.tracks[t_i]["lost"] += 1

        # Khởi tạo track mới từ các detection confidence cao chưa khớp
        for d_i in unm_high:
            real_i = high_idx[d_i]
            new_tr = {
                "tid": self.next_id,
                "box": boxes[real_i].copy(),
                "vel": np.zeros(4, dtype=np.float32),
                "lost": 0,
                "conf": confs[real_i],
                "cls": clss[real_i],
            }
            self.next_id += 1
            self.tracks.append(new_tr)
            active_outputs.append(new_tr)

        # Lọc bỏ các track đã mất dấu quá lâu
        self.tracks = [tr for tr in self.tracks if tr["lost"] <= self.track_buffer]

        if not active_outputs:
            return SimpleDetections(
                np.zeros((0, 4), np.float32),
                np.zeros(0, np.float32),
                np.zeros(0, int),
                np.zeros(0, int)
            )

        out_xyxy = np.array([tr["box"] for tr in active_outputs], dtype=np.float32)
        out_conf = np.array([tr["conf"] for tr in active_outputs], dtype=np.float32)
        out_cls = np.array([tr["cls"] for tr in active_outputs], dtype=int)
        out_tid = np.array([tr["tid"] for tr in active_outputs], dtype=int)
        return SimpleDetections(out_xyxy, out_conf, out_cls, out_tid)


def make_tracker(fps):
    buffer_memory = int(fps * 30)
    try:
        import supervision as sv
        if hasattr(sv, "ByteTrack"):
            try:
                return sv.ByteTrack(track_activation_threshold=0.45, lost_track_buffer=buffer_memory,
                                    minimum_matching_threshold=0.8, frame_rate=int(fps))
            except TypeError:
                return sv.ByteTrack(track_thresh=0.45, track_buffer=buffer_memory,
                                    match_thresh=0.8, frame_rate=int(fps))
    except Exception:
        pass
    return BuiltinByteTrack(track_thresh=0.45, track_buffer=buffer_memory, match_thresh=0.8)


def make_detections(xyxy, confidence, class_id):
    try:
        import supervision as sv
        if hasattr(sv, "ByteTrack") and hasattr(sv, "Detections"):
            return sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_id)
    except Exception:
        pass
    return SimpleDetections(xyxy=xyxy, confidence=confidence, class_id=class_id)


# ============================ VẼ (TRONG SUỐT, CHỮ NHỎ) ============================
def color_for(tid):
    rng = np.random.RandomState(tid * 7919 % 2 ** 31)
    return tuple(int(c) for c in rng.randint(60, 255, 3))


def draw(frame, box, track, shot, fps):
    x1, y1, x2, y2 = [int(v) for v in box]
    col = color_for(track.tid + 1000 * track.shot)

    cv2.rectangle(frame, (x1, y1), (x2, y2), col, 1)

    label = f"#{track.tid} {track.speed_ms * 3.6:4.1f}km/h {track.dist_m:5.1f}m"
    font_scale = 0.35
    thickness = 1

    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
    ty = max(y1 - 4, th + 4)

    cv2.putText(frame, label, (x1, ty - 2), cv2.FONT_HERSHEY_SIMPLEX, font_scale,
                (0, 255, 255), thickness, cv2.LINE_AA)


# ============================ CHƯƠNG TRÌNH CHÍNH ============================
def build_report(tracks, fps):
    rows = []
    for t in tracks.values():
        seen = t.frames_seen / fps
        if seen < MIN_TRACK_SEC:
            continue
        measured = t.time_moving + t.time_still
        rows.append({
            "shot": t.shot,
            "player_id": t.tid,
            "start_s": round(t.first_frame / fps, 2),
            "end_s": round(t.last_frame / fps, 2),
            "time_visible_s": round(max(seen, measured), 2),
            "time_moving_s": round(t.time_moving, 2),
            "time_still_s": round(t.time_still, 2),
            "distance_m": round(t.dist_m, 2),
            "avg_speed_kmh": round(t.dist_m / measured * 3.6, 2) if measured > 0 else 0.0,
            "max_speed_kmh": round(t.max_speed_ms * 3.6, 2),
            "sprint_time_s": round(t.sprint_time, 2),
        })
    cols = ["shot", "player_id", "start_s", "end_s", "time_visible_s", "time_moving_s",
            "time_still_s", "distance_m", "avg_speed_kmh", "max_speed_kmh", "sprint_time_s"]
    return pd.DataFrame(rows, columns=cols)


def save_chart(df, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    if df.empty:
        return
    top = df.sort_values("distance_m", ascending=False).head(15).iloc[::-1]
    labels = [f"S{s}-#{p}" for s, p in zip(top["shot"], top["player_id"])]
    plt.figure(figsize=(8, 5))
    plt.barh(labels, top["distance_m"], color="#2e7d32")
    plt.xlabel("Quãng đường (m)")
    plt.title("Top 15 track có quãng đường dài nhất (S = cảnh, # = ID)")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


def reencode_h264(path):
    if shutil.which("ffmpeg") is None:
        return
    tmp = path + ".tmp.mp4"
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-vcodec", "libx264",
                        "-pix_fmt", "yuv420p", "-crf", "20", tmp])
    if r.returncode == 0:
        os.replace(tmp, path)


def parse_args(args_list=None):
    ap = argparse.ArgumentParser(description="Thống kê di chuyển cầu thủ bóng đá")
    ap.add_argument("--source", required=True, help="video đầu vào")
    ap.add_argument("--output", default="output_processed.mp4", help="video kết quả")
    ap.add_argument("--csv", default=None, help="file CSV thống kê")
    ap.add_argument("--model", default="yolov8m.pt", help="yolov8n/s/m/l/x.pt")
    ap.add_argument("--imgsz", type=int, default=1280, help="kích thước ảnh")
    ap.add_argument("--conf", type=float, default=0.15, help="ngưỡng confidence")
    ap.add_argument("--tiles", action="store_true", help="cắt ô 2x2")
    ap.add_argument("--device", default=None, help="cpu/gpu")
    ap.add_argument("--max-frames", type=int, default=0, help="số khung hình xử lý")
    ap.add_argument("--max-seconds", type=float, default=0, help="số giây tối đa xử lý")
    ap.add_argument("--show", action="store_true", help="hiển thị cửa sổ")

    if args_list is not None:
        return ap.parse_args(args_list)
    return ap.parse_args()


def main(args_list=None):
    args = parse_args(args_list)
    cap = cv2.VideoCapture(args.source)
    if not cap.isOpened():
        sys.exit(f"Không mở được video: {args.source}")
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    if args.max_seconds > 0:
        args.max_frames = int(args.max_seconds * fps)

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if args.max_frames:
        total = min(total, args.max_frames)

    print(f"Video: {W}x{H}, {fps:.0f} fps, {total} frame. Đang nạp model...", flush=True)
    writer = cv2.VideoWriter(args.output, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    detector = PersonDetector(args.model, args.imgsz, args.conf, 0.5, args.device, args.tiles)
    cutter, cam = CutDetector(), CameraMotion(W, H)
    tracker = make_tracker(fps)

    id_manager = IdentityManager(max_players=22, match_thresh=0.55)
    color_matcher = DynamicTeamColorMatcher()
    first_frames_shirts = []

    print("Bắt đầu xử lý frame... (nhấn Ctrl+C để dừng)", flush=True)
    tracks = {}
    shot = 1
    t0 = time.time()
    idx = 0

    # Bước 1: Trích xuất nhanh dữ liệu màu áo từ 15 frame đầu để học
    while idx < min(15, total):
        ok, frame = cap.read()
        if not ok:
            break
        boxes, scores = detector.detect(frame)
        boxes, scores = filter_detections(boxes, scores, frame)

        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        for (x1, y1, x2, y2) in boxes:
            w, h = x2 - x1, y2 - y1
            cx1, cx2 = int(x1 + w * 0.2), int(x2 - w * 0.2)
            cy1, cy2 = int(y1 + h * 0.1), int(y1 + h * 0.5)
            cy1, cy2 = max(0, cy1), max(0, cy2)
            cx1, cx2 = max(0, cx1), max(0, cx2)
            shirt_hsv = hsv_frame[cy1:cy2, cx1:cx2]
            if shirt_hsv.size > 0:
                first_frames_shirts.append(shirt_hsv)
        idx += 1

    # Thực hiện gom cụm màu áo động
    color_matcher.learn_dominant_colors(first_frames_shirts)

    # Reset lại con trỏ video để xử lý từ đầu
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    idx = 0

    # Bước 2: Chạy xử lý và theo dõi chính thức
    while idx < total:
        ok, frame = cap.read()
        if not ok:
            break

        boxes, scores = detector.detect(frame)
        boxes, scores = filter_detections(boxes, scores, frame)

        if len(boxes) > 0:
            keep_color = filter_players_by_dynamic_color(frame, boxes, color_matcher)
            boxes = boxes[keep_color]
            scores = scores[keep_color]

        if cutter.update(frame) and idx > 0:
            shot += 1
            cam.reset()
            tracker = make_tracker(fps)
            id_manager = IdentityManager(max_players=22, match_thresh=0.55)

        cam.update(frame, boxes)
        boxes_ref = cam.boxes_to_ref(boxes)

        dets = make_detections(
            xyxy=boxes_ref.reshape(-1, 4).astype(np.float32),
            confidence=scores.astype(np.float32),
            class_id=np.zeros(len(boxes_ref), dtype=int)
        )
        tracked = tracker.update_with_detections(dets)

        if len(tracked) > 0 and tracked.tracker_id is not None:
            boxes_now = cam.boxes_from_ref(tracked.xyxy)

            bt_ids = [int(tid) for tid in tracked.tracker_id]
            id_mapping = id_manager.process_frame(frame, bt_ids, boxes_now)

            for b_ref, b_now, tid in zip(tracked.xyxy, boxes_now, tracked.tracker_id):
                bt_id = int(tid)

                if bt_id not in id_mapping or id_mapping[bt_id] is None:
                    continue

                global_id = id_mapping[bt_id]

                key = (shot, global_id)
                if key not in tracks:
                    tracks[key] = PlayerTrack(shot, global_id, idx)
                tr = tracks[key]
                tr.update(idx, b_ref, b_now, (W, H), fps)

                tr.tid = global_id
                draw(frame, b_now, tr, shot, fps)

        cv2.putText(frame, f"Shot {shot} | {idx / fps:6.1f}s", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(frame)
        if args.show:
            cv2.imshow("tracking", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        idx += 1
        if idx % 10 == 0 or idx == 1:
            el = time.time() - t0
            print(f"[{idx}/{total}] {idx / el:.2f} fps, còn ~{(total - idx) / (idx / el) / 60:.1f} phút",
                  flush=True)

    cap.release()
    writer.release()
    cv2.destroyAllWindows()

    df = build_report(tracks, fps)
    csv_path = args.csv or os.path.splitext(args.output)[0] + "_stats.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    save_chart(df, os.path.splitext(args.output)[0] + "_chart.png")
    reencode_h264(args.output)

    print(f"\nĐã xử lý {idx} frame, {shot} cảnh, {len(df)} track hợp lệ.")
    print(f"Video : {args.output}\nCSV   : {csv_path}")
    if not df.empty:
        print(df.sort_values("distance_m", ascending=False).head(10).to_string(index=False))

if __name__ == "__main__":
    main([
        '--source', '/bongda.mp4',
        '--output', '/content/output.mp4',
    ])

Video: 848x480, 30 fps, 6350 frame. Đang nạp model...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Bắt đầu xử lý frame... (nhấn Ctrl+C để dừng)

[Bộ Nhận Diện Màu Động] Đã học tự động thành công dải màu của 2 đội bóng.
[1/6350] 0.11 fps, còn ~1001.0 phút
[10/6350] 0.96 fps, còn ~109.5 phút
[20/6350] 1.76 fps, còn ~59.9 phút
[30/6350] 2.46 fps, còn ~42.9 phút
[40/6350] 3.09 fps, còn ~34.1 phút
[50/6350] 3.65 fps, còn ~28.8 phút
[60/6350] 4.15 fps, còn ~25.3 phút
[70/6350] 4.59 fps, còn ~22.8 phút
[80/6350] 4.99 fps, còn ~20.9 phút
[90/6350] 5.35 fps, còn ~19.5 phút
[100/6350] 5.67 fps, còn ~18.4 phút
[110/6350] 5.97 fps, còn ~17.4 phút
[120/6350] 6.24 fps, còn ~16.7 phút
[130/6350] 6.49 fps, còn ~16.0 phút
[140/6350] 6.73 fps, còn ~15.4 phút
[1